In [42]:
import pandas as pd
import numpy as np
import re
import nltk

In [43]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mansi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mansi\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Mansi\AppData\Roaming\nltk_data...


True

In [44]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [45]:
df = pd.read_csv("../data/cleaned_reviews.csv")

In [46]:
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience,Sentiment
0,Lisa,/users/63c2d28aff751c001470ce46,US,1 review,2023-01-14T18:41:59.000Z,1,What happened to quality customer service?,"I ordered several items on Jan 2, this review ...","January 14, 2023",Negative
1,J Ellery,/users/4f5291a30000640001153f96,GB,6 reviews,2012-03-03T21:50:56.000Z,5,Best on the web,Never any problems.goods always arrive when st...,"March 03, 2012",Positive
2,Heather Weber,/users/5d37078e6614abe3ab453adf,US,1 review,2019-07-23T13:12:41.000Z,3,Issues with online purchases,"Recently, I have been having issues with my or...","July 23, 2019",Neutral
3,Mrs Lowe,/users/5666ab6d0000ff0001f14958,GB,2 reviews,2015-12-08T10:07:54.695Z,5,Excellent as always,Very good.,NaN,Positive
4,Lolo Paz,/users/5d8732fb681610d90db66d2e,US,2 reviews,2019-09-22T08:41:50.000Z,4,Ive been shopping with amazon for…,Ive been shopping with amazon for almost a yea...,"September 22, 2019",Positive


In [47]:
df.shape

(42740, 10)

In [48]:
df.columns

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience', 'Sentiment'],
      dtype='object')

In [49]:
stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

In [50]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [
        word for word in words
        if word not in stop_words
    ]

    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

In [51]:
df['Clean_Review'] = df['Review Text'].astype(str).apply(clean_text)

In [52]:
df[['Review Text', 'Clean_Review']].head()

,Review Text,Clean_Review
0,"I ordered several items on Jan 2, this review ...",ordered several item jan review reference retu...
1,Never any problems.goods always arrive when st...,never problemsgoods always arrive stated exell...
2,"Recently, I have been having issues with my or...",recently issue order receiving item order rece...
3,Very good.,good
4,Ive been shopping with amazon for almost a yea...,ive shopping amazon almost year get delivery a...


In [53]:
df['Clean_Review'].isnull().sum()

np.int64(0)

In [54]:
df = df[df['Clean_Review'].str.strip() != ""]

In [55]:
from sklearn.preprocessing import LabelEncoder

In [56]:
encoder = LabelEncoder()

df['Sentiment_Label'] = encoder.fit_transform(
    df['Sentiment']
)

In [57]:
df[['Sentiment', 'Sentiment_Label']].head()

,Sentiment,Sentiment_Label
0,Negative,0
1,Positive,2
2,Neutral,1
3,Positive,2
4,Positive,2


In [58]:
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
label_mapping

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}

In [59]:
df.to_csv(
    "../data/processed_reviews.csv",
    index=False
)

In [60]:
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience,Sentiment,Clean_Review,Sentiment_Label
0,Lisa,/users/63c2d28aff751c001470ce46,US,1 review,2023-01-14T18:41:59.000Z,1,What happened to quality customer service?,"I ordered several items on Jan 2, this review ...","January 14, 2023",Negative,ordered several item jan review reference retu...,0
1,J Ellery,/users/4f5291a30000640001153f96,GB,6 reviews,2012-03-03T21:50:56.000Z,5,Best on the web,Never any problems.goods always arrive when st...,"March 03, 2012",Positive,never problemsgoods always arrive stated exell...,2
2,Heather Weber,/users/5d37078e6614abe3ab453adf,US,1 review,2019-07-23T13:12:41.000Z,3,Issues with online purchases,"Recently, I have been having issues with my or...","July 23, 2019",Neutral,recently issue order receiving item order rece...,1
3,Mrs Lowe,/users/5666ab6d0000ff0001f14958,GB,2 reviews,2015-12-08T10:07:54.695Z,5,Excellent as always,Very good.,NaN,Positive,good,2
4,Lolo Paz,/users/5d8732fb681610d90db66d2e,US,2 reviews,2019-09-22T08:41:50.000Z,4,Ive been shopping with amazon for…,Ive been shopping with amazon for almost a yea...,"September 22, 2019",Positive,ive shopping amazon almost year get delivery a...,2
